# Chapter 2. Molecular Mechanics

Part 1: measuring and changing molecular geometry

## 2.1. Molecular Geometry

A molecular graph tells us **which atoms are connected**; a conformer stores **where those atoms are**. A molecule may have several conformers with the same connectivity.

**Learning objectives**

By the end of this notebook, you should be able to:

- Calculate distances, bond angles, and signed dihedral angles from Cartesian coordinates.
- Check your NumPy calculations against RDKit.
- Explain atom ordering, angle units, and geometries for which an angle is undefined.
- Change coordinates on a copy without mistaking the change for an energy minimization.

**Before you start:** use the environment in the repository README, then **Restart Kernel and Run All Cells**. This notebook is independent of other notebooks. All calculations and Matplotlib figures work offline. The final interactive viewer is optional.

**Conventions:** atom indices start at **0**. Distances are in angstroms ($1\ \mathrm{\AA}=10^{-10}\ \mathrm{m}$); NumPy trigonometric functions use radians. RDKit methods ending in `Deg` use degrees. These are isolated-molecule coordinates, so no periodic-boundary corrections are needed.

### A bridge from a drawing to a measurement

A bond line in a 2D drawing records a connection. To measure a length, each atom needs a position $(x,y,z)$ in a common coordinate system. Subtracting two positions gives a **displacement vector**: “how far in each direction to go from one atom to the other.” Its **norm** is its length.

| Internal coordinate | Atoms needed | Question |
| --- | --- | --- |
| Distance $r_{ij}$ | Two | How far apart are these atoms? |
| Bond angle $\theta_{ijk}$ | Three, with $j$ at the center | How wide is the opening between two bonds? |
| Dihedral $\phi_{ijkl}$ | Four, around $j$–$k$ | How are two groups twisted relative to each other? |

These are **internal** coordinates because translating or rotating the whole molecule leaves them unchanged. Atom indices are labels in an array, not chemical names; preserve the mapping when comparing files.

**Core route:** inspect ethane → calculate one distance and one angle → rotate a group → read the projection and distance-change figures. The cross products and `atan2` derivation are **deeper mathematical detail**; use the tested helper first and return to the derivation when comfortable with vectors. A cross product constructs a vector perpendicular to a plane; a dot product measures directional agreement.

**Predict:** will moving every atom 10 angstroms to the right change the C–C bond length? Will rotating only one methyl group change distances between its hydrogens and the other group's hydrogens?

### 2.1.1. Bond Length

The bond length $r_{ij}$ is the distance between the nuclei of bonded atoms $i$ and $j$:

$$r_{ij}=\lVert\mathbf r_j-\mathbf r_i\rVert
=\sqrt{(x_j-x_i)^2+(y_j-y_i)^2+(z_j-z_i)^2}.$$

The same equation measures a distance between **any** two atoms. It does not establish whether a bond exists. A bond length belongs to a particular geometry; it need not equal an experimental average or a force field's equilibrium value.

We use the original ethane coordinates below as a fixed teaching example. XYZ stores an atom count, a comment, and element/coordinate rows; it does **not** store bonds. Here the carbon atoms are rows 0 and 1; hydrogens 2--4 belong to carbon 0, and hydrogens 5--7 to carbon 1.

In [ ]:
# XYZ string for ethane (C2H6)
ethane_xyz = """8
Ethane
C      -0.765806   -0.000316   0.000000
C      0.765806    0.000316    0.000000
H      -1.165351   1.040005    -0.000000
H      -1.164581   -0.520796   0.901055
H      -1.164581   -0.520796   -0.901055
H      1.165351    -1.040005   0.000000
H      1.164581    0.520796    0.901055
H      1.164581    0.520796    -0.901055
"""

For the carbon atoms in this XYZ block:

$$r_{\mathrm{CC}}=\sqrt{(1.531612)^2+(0.000632)^2}
\approx 1.531612\ \mathrm{\AA}.$$

Reporting more digits than the input coordinates justify would imply false precision.

Build the graph from SMILES and attach the supplied coordinates. No random embedding is needed when coordinates already exist. The XYZ rows and graph atoms **must correspond in the same order**. Checking element labels catches some mistakes, but cannot distinguish a permutation of identical atoms; the mapping must be established from the data source.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from rdkit import Chem, rdBase
from rdkit.Chem import rdMolTransforms

print(f"RDKit {rdBase.rdkitVersion}; NumPy {np.__version__}")


def plot_geometry(mol, title, ax=None):
    """Plot the actual 3D coordinates with zero-based atom indices, offline."""
    if ax is None:
        fig = plt.figure(figsize=(7, 5))
        ax = fig.add_subplot(111, projection="3d")
        fig.subplots_adjust(left=0.03, right=0.86, bottom=0.08, top=0.90)
    xyz = mol.GetConformer().GetPositions()
    for bond in mol.GetBonds():
        pair = xyz[[bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()]]
        ax.plot(*pair.T, color="0.45", linewidth=2)
    for atom, point in zip(mol.GetAtoms(), xyz):
        color = "#334155" if atom.GetSymbol() == "C" else "#93c5fd"
        ax.scatter(*point, color=color, s=80)
        ax.text(*point, f" {atom.GetSymbol()}{atom.GetIdx()}", fontsize=9)
    center = xyz.mean(axis=0)
    half_width = max(np.ptp(xyz, axis=0).max() / 2, 0.5) + 0.3
    ax.set(xlim=(center[0]-half_width, center[0]+half_width),
           ylim=(center[1]-half_width, center[1]+half_width),
           zlim=(center[2]-half_width, center[2]+half_width),
           xlabel="x (\u00c5)", ylabel="y (\u00c5)",
           zlabel="", title=title)
    ax.set_box_aspect((1, 1, 1))
    ax.text2D(1.04, 0.5, "z (\u00c5)", transform=ax.transAxes, rotation=90, va="center")
    return ax

In [ ]:
xyz_mol = Chem.MolFromXYZBlock(ethane_xyz)
ethane_mol = Chem.AddHs(Chem.MolFromSmiles("CC"))
if xyz_mol is None:
    raise ValueError("The XYZ block could not be read.")
if [a.GetSymbol() for a in xyz_mol.GetAtoms()] != [a.GetSymbol() for a in ethane_mol.GetAtoms()]:
    raise ValueError("XYZ elements/order do not match the molecular graph.")

# This mapping is known for this particular XYZ block, not for arbitrary XYZ files.
assert sorted(a.GetIdx() for a in ethane_mol.GetAtomWithIdx(0).GetNeighbors()) == [1, 2, 3, 4]
assert sorted(a.GetIdx() for a in ethane_mol.GetAtomWithIdx(1).GetNeighbors()) == [0, 5, 6, 7]
ethane_mol.AddConformer(Chem.Conformer(xyz_mol.GetConformer()), assignId=True)
reference_xyz = ethane_mol.GetConformer().GetPositions().copy()
plot_geometry(ethane_mol, "Ethane: supplied coordinates and atom indices")
plt.show()

In [ ]:
xyz = ethane_mol.GetConformer().GetPositions()
cc_distance_numpy = np.linalg.norm(xyz[1] - xyz[0])
ch_distance_numpy = np.linalg.norm(xyz[2] - xyz[0])
print(f"C0-C1 distance: {cc_distance_numpy:.6f} angstrom")
print(f"C0-H2 distance: {ch_distance_numpy:.6f} angstrom")

RDKit provides the same measurements through [`rdMolTransforms`](https://www.rdkit.org/docs/source/rdkit.Chem.rdMolTransforms.html). Import that module explicitly so the code does not depend on another import loading it as a side effect.

In [ ]:
conf = ethane_mol.GetConformer()
cc_distance_rdkit = rdMolTransforms.GetBondLength(conf, 0, 1)
ch_distance_rdkit = rdMolTransforms.GetBondLength(conf, 0, 2)
np.testing.assert_allclose([cc_distance_numpy, ch_distance_numpy],
                           [cc_distance_rdkit, ch_distance_rdkit], atol=1e-12)
print(f"RDKit agrees: C-C = {cc_distance_rdkit:.6f}; C-H = {ch_distance_rdkit:.6f} angstrom")

`SetBondLength` changes coordinates, including the connected atoms on the moved side. It does not recalculate bonding or find a stable geometry. Make a copy before editing, so later measurements still refer to the original structure. This deliberately doubled bond is an **unphysical demonstration**, not a prediction of stretched ethane or bond dissociation.

In [ ]:
stretched_ethane = Chem.Mol(ethane_mol)
new_cc_distance = 2 * cc_distance_rdkit
rdMolTransforms.SetBondLength(stretched_ethane.GetConformer(), 0, 1, new_cc_distance)
plot_geometry(stretched_ethane, "Coordinate edit: doubled C-C distance")
plt.show()

In [ ]:
measured = rdMolTransforms.GetBondLength(stretched_ethane.GetConformer(), 0, 1)
np.testing.assert_allclose(measured, new_cc_distance, atol=1e-10)
np.testing.assert_allclose(ethane_mol.GetConformer().GetPositions(), reference_xyz)
print(f"Edited distance: {measured:.6f} angstrom; original molecule unchanged.")

### 2.1.2. Bond Angle

For atoms $i$--$j$--$k$, the bond angle is centered on **$j$**. Both vectors must point outward from this central atom. A bond angle lies between $0^\circ$ and $180^\circ$.

With $\mathbf u=\mathbf r_i-\mathbf r_j$ and $\mathbf v=\mathbf r_k-\mathbf r_j$:

$$\theta=\arccos\!\left(\frac{\mathbf u\cdot\mathbf v}{\lVert\mathbf u\rVert\lVert\mathbf v\rVert}\right).$$

The angle is undefined when either vector has zero length. Roundoff may put the cosine slightly outside $[-1,1]$, so clip it before calling `arccos`.

Calculate the H2--C0--C1 angle using NumPy:

In [ ]:
def bond_angle_deg(p_i, p_j, p_k):
    u = np.asarray(p_i, dtype=float) - np.asarray(p_j, dtype=float)
    v = np.asarray(p_k, dtype=float) - np.asarray(p_j, dtype=float)
    u_norm, v_norm = np.linalg.norm(u), np.linalg.norm(v)
    if u_norm < 1e-12 or v_norm < 1e-12:
        raise ValueError("An angle needs two nonzero bond vectors.")
    cosine = np.dot(u / u_norm, v / v_norm)
    return float(np.degrees(np.arccos(np.clip(cosine, -1.0, 1.0))))


hcc_angle_numpy = bond_angle_deg(xyz[2], xyz[0], xyz[1])
print(f"H2-C0-C1 angle: {hcc_angle_numpy:.3f} degrees")

Check the central-atom ordering by comparing with RDKit:

In [ ]:
hcc_angle_rdkit = rdMolTransforms.GetAngleDeg(ethane_mol.GetConformer(), 2, 0, 1)
np.testing.assert_allclose(hcc_angle_numpy, hcc_angle_rdkit, atol=1e-10)
print(f"RDKit angle: {hcc_angle_rdkit:.3f} degrees")

In [ ]:
# Draw the two actual displacement vectors in their common plane.
angle_points = xyz[[2, 0, 1]]
origin = angle_points[1]
u = angle_points[0] - origin
v = angle_points[2] - origin
unit_u = u / np.linalg.norm(u)
normal = np.cross(u, v)
normal = normal / np.linalg.norm(normal)
unit_perpendicular = np.cross(normal, unit_u)
angle_plane = np.column_stack(((angle_points-origin) @ unit_u,
                                (angle_points-origin) @ unit_perpendicular))
fig, ax = plt.subplots(figsize=(6, 4), constrained_layout=True)
for endpoint, label, color in [(angle_plane[0], "H2", "#2563eb"),
                              (angle_plane[2], "C1", "#b45309")]:
    ax.annotate("", xy=endpoint, xytext=(0, 0),
                arrowprops={"arrowstyle": "->", "color": color, "lw": 2})
    ax.text(*(endpoint * 1.12), label, ha="center", color=color)
ax.scatter([0], [0], color="#334155", zorder=3)
ax.text(0, -0.15, "C0: central atom", ha="center")
arc_angles = np.linspace(0, np.deg2rad(hcc_angle_numpy), 80)
ax.plot(0.35*np.cos(arc_angles), 0.35*np.sin(arc_angles), color="#334155")
ax.set(xlabel="Coordinate along C0-H2 (angstrom)",
       ylabel="Perpendicular coordinate in the plane (angstrom)",
       title=f"Two vectors with one origin: H2-C0-C1 = {hcc_angle_numpy:.1f} degrees",
       xlim=(-0.8, 1.5), ylim=(-0.35, 1.8), aspect="equal")
plt.show()

Change the angle on a fresh copy. A value of $120^\circ$ demonstrates the operation but is not an optimized ethane angle. In a ring, moving an internal coordinate can be constrained by ring closure; do not assume these setters work for every bonded tuple.

In [ ]:
bent_ethane = Chem.Mol(ethane_mol)
new_hcc_angle = 120.0
rdMolTransforms.SetAngleDeg(bent_ethane.GetConformer(), 2, 0, 1, new_hcc_angle)
plot_geometry(bent_ethane, "Coordinate edit: H2-C0-C1 = 120 degrees")
plt.show()

In [ ]:
measured = rdMolTransforms.GetAngleDeg(bent_ethane.GetConformer(), 2, 0, 1)
np.testing.assert_allclose(measured, new_hcc_angle, atol=1e-10)
print(f"Edited angle: {measured:.3f} degrees")

### 2.1.3. Torsion Angle

A torsion (dihedral) for $i$--$j$--$k$--$l$ describes the relative orientation of the planes $(i,j,k)$ and $(j,k,l)$ around the middle bond $j$--$k$. Use a **signed** angle to distinguish the two rotation directions. Its sign depends on the convention, so state the atom order and compare like conventions.

For a chosen H--C--C--H tuple in ethane, $0^\circ$ is eclipsed and $60^\circ$ is staggered. Other hydrogen choices give angles offset by roughly $120^\circ$ for the same conformer.

**Picture the viewing direction:** look from atom $j$ toward atom $k$, along the central bond. The projected $j$–$i$ and $k$–$l$ bonds rotate relative to one another. Later in this section we generate this view directly from the ethane coordinates, so the diagram can be checked against the numbers.

One signed convention, used by the implementation below and checked against RDKit, is:

$$\begin{aligned}
\mathbf b_1&=\mathbf r_j-\mathbf r_i,&
\mathbf b_2&=\mathbf r_k-\mathbf r_j,&
\mathbf b_3&=\mathbf r_l-\mathbf r_k,\\
\mathbf n_1&=\mathbf b_1\times\mathbf b_2,&
\mathbf n_2&=\mathbf b_2\times\mathbf b_3,\\
\phi&=\operatorname{atan2}\!\left[
\widehat{\mathbf b}_2\cdot(\mathbf n_1\times\mathbf n_2),
\mathbf n_1\cdot\mathbf n_2\right].
\end{aligned}$$

Here $\widehat{\mathbf b}_2=\mathbf b_2/\lVert\mathbf b_2\rVert$ and `atan2(y, x)` preserves the sign. The dihedral is undefined if a bond vector is zero or if either defining plane collapses because three atoms are collinear. Angles differing by $360^\circ$ are equivalent; $+180^\circ$ and $-180^\circ$ meet at the same branch cut.

Calculate the H2--C0--C1--H6 dihedral with the same atom order in both implementations:

In [ ]:
def dihedral_deg(p_i, p_j, p_k, p_l):
    points = np.asarray([p_i, p_j, p_k, p_l], dtype=float)
    bonds = np.diff(points, axis=0)
    lengths = np.linalg.norm(bonds, axis=1)
    if np.any(lengths < 1e-12):
        raise ValueError("A dihedral needs nonzero bond vectors.")
    b1, b2, b3 = bonds / lengths[:, None]
    n1, n2 = np.cross(b1, b2), np.cross(b2, b3)
    if min(np.linalg.norm(n1), np.linalg.norm(n2)) < 1e-12:
        raise ValueError("A dihedral is undefined for collinear atoms.")
    y = np.dot(b2, np.cross(n1, n2))
    x = np.dot(n1, n2)
    return float(np.degrees(np.arctan2(y, x)))


torsion_indices = (2, 0, 1, 6)
torsion_numpy = dihedral_deg(*xyz[list(torsion_indices)])
print(f"H2-C0-C1-H6 torsion: {torsion_numpy:.3f} degrees")

Compare angles with a wrapped difference so that the two representations of 180 degrees count as equal:

In [ ]:
def angle_difference_deg(a, b):
    return (a - b + 180.0) % 360.0 - 180.0


torsion_rdkit = rdMolTransforms.GetDihedralDeg(ethane_mol.GetConformer(), *torsion_indices)
assert abs(angle_difference_deg(torsion_numpy, torsion_rdkit)) < 1e-10
print(f"RDKit torsion: {torsion_rdkit:.3f} degrees")

Rotate the group on the far side of C0--C1 using a **single** dihedral setter. There is no need to set a separate torsion for every hydrogen on that group. As with the other setters, this changes coordinates without optimizing the energy.

In [ ]:
rotated_ethane = Chem.Mol(ethane_mol)
new_torsion = 45.0
rdMolTransforms.SetDihedralDeg(rotated_ethane.GetConformer(), *torsion_indices, new_torsion)
plot_geometry(rotated_ethane, "Coordinate edit: H2-C0-C1-H6 = 45 degrees")
plt.show()

In [ ]:
measured = rdMolTransforms.GetDihedralDeg(rotated_ethane.GetConformer(), *torsion_indices)
assert abs(angle_difference_deg(measured, new_torsion)) < 1e-10
np.testing.assert_allclose(ethane_mol.GetConformer().GetPositions(), reference_xyz)
print(f"Edited torsion: {measured:.3f} degrees; original molecule unchanged.")

### See the torsion by looking down the central bond

This **Newman-style projection** views C0 toward C1. Blue bonds start at the front carbon; orange bonds belong to the rear carbon (drawn as a circle). Atom labels connect the picture to the array indices. Each projected bond is scaled to a common drawing length; its **direction**, not its drawn length, comes from the 3D coordinates.

Find H2 and H6 in both panels. The selected H2–C0–C1–H6 angle changes, while the attached methyl groups retain their internal geometry.

In [ ]:
def draw_bond_projection(molecule, ax, title):
    points = molecule.GetConformer().GetPositions()
    axis = points[1] - points[0]
    axis = axis / np.linalg.norm(axis)
    reference = points[2] - points[0]
    reference = reference - np.dot(reference, axis) * axis
    horizontal = reference / np.linalg.norm(reference)
    vertical = np.cross(axis, horizontal)
    ax.add_patch(plt.Circle((0, 0), 0.20, fill=False, color="#b45309", linewidth=2))
    for center, color in [(1, "#b45309"), (0, "#2563eb")]:
        for atom in molecule.GetAtomWithIdx(center).GetNeighbors():
            if atom.GetAtomicNum() != 1:
                continue
            vector = points[atom.GetIdx()] - points[center]
            projection = np.array([np.dot(vector, horizontal), np.dot(vector, vertical)])
            projection = projection / np.linalg.norm(projection)
            start = 0.20 * projection if center == 1 else np.zeros(2)
            end = (1.05 if center == 1 else 0.86) * projection
            ax.plot([start[0], end[0]], [start[1], end[1]], color=color, lw=2)
            ax.text(*(end * 1.14), f"H{atom.GetIdx()}", ha="center", va="center", color=color)
    ax.scatter([0], [0], color="#2563eb", s=35, zorder=4)
    ax.set(xlim=(-1.4, 1.4), ylim=(-1.4, 1.4), aspect="equal", title=title)
    ax.axis("off")

fig, axes = plt.subplots(1, 2, figsize=(9, 4), constrained_layout=True)
for molecule, ax, title in [(ethane_mol, axes[0], "Original coordinates"),
                             (rotated_ethane, axes[1], "One group rotated: selected torsion 45 degrees")]:
    draw_bond_projection(molecule, ax, title)
fig.suptitle("View C0 toward C1: blue front, orange rear; directions from coordinates")
plt.show()

### Research application: did a conformer edit change local bonds or contacts?

Before passing an edited geometry to a calculation, a researcher checks what moved. A **distance matrix** stores all pair distances: entry `(i, j)` is the distance between atoms `i` and `j`. Compare the original and rotated ethane matrices. Blue means the pair moved closer; red means it moved farther apart.

**Predict:** which blocks should remain near zero? The diagonal compares each atom with itself, so it must always be zero. This is a coordinate calculation, not a contact-energy or steric-clash model.

In [ ]:
def pair_distances(points):
    differences = points[:, None, :] - points[None, :, :]
    return np.linalg.norm(differences, axis=-1)

distance_before = pair_distances(ethane_mol.GetConformer().GetPositions())
distance_after = pair_distances(rotated_ethane.GetConformer().GetPositions())
distance_change = distance_after - distance_before
bond_changes = np.array([distance_change[b.GetBeginAtomIdx(), b.GetEndAtomIdx()]
                         for b in ethane_mol.GetBonds()])
np.testing.assert_allclose(bond_changes, 0, atol=1e-8)
np.testing.assert_allclose(distance_change, distance_change.T, atol=1e-12)
assert np.max(np.abs(distance_change)) > 0.01
fig, ax = plt.subplots(figsize=(6, 4.6), constrained_layout=True)
limit = np.max(np.abs(distance_change))
heatmap = ax.imshow(distance_change, cmap="RdBu_r", vmin=-limit, vmax=limit)
atom_labels = [f"{atom.GetSymbol()}{atom.GetIdx()}" for atom in ethane_mol.GetAtoms()]
ax.set(xticks=range(8), xticklabels=atom_labels, yticks=range(8), yticklabels=atom_labels,
       xlabel="Atom j", ylabel="Atom i", title="Pair-distance change after the torsion edit")
fig.colorbar(heatmap, ax=ax, label="After minus before (angstrom)")
plt.show()
print(f"Largest bond-length change: {np.max(np.abs(bond_changes)):.2e} angstrom")
print(f"Largest pair-distance change: {limit:.3f} angstrom")

**Decision:** the coordinate setter preserved bond lengths here while changing some nonbonded separations. This is the intended local edit; it does not establish that the new conformer is energetically favorable. Chapter 2 Part 2 supplies that additional model.

**Guided exercise:** translate the whole original molecule and recompute its distance matrix. **Selected answer:** every entry is unchanged within floating-point tolerance. A mirror reflection also preserves pair distances, so a distance matrix alone cannot distinguish opposite handedness. The signed dihedral supplies information that distances discard.

### 2.1.4. Self-checks and exercises

The following checks test known geometries and invariance to translating/rotating the whole molecule. Such transformations should not change internal coordinates.

In [ ]:
assert np.isclose(bond_angle_deg([1, 0, 0], [0, 0, 0], [0, 1, 0]), 90)
# A known positive dihedral and its mirror test the sign, not only the magnitude.
assert np.isclose(dihedral_deg([0, 1, 0], [0, 0, 0], [1, 0, 0], [1, 0, 1]), 90)
assert np.isclose(dihedral_deg([0, 1, 0], [0, 0, 0], [1, 0, 0], [1, 0, -1]), -90)
rotation = np.array([[0, -1, 0], [1, 0, 0], [0, 0, 1]])
moved_xyz = xyz @ rotation.T + np.array([10.0, -4.0, 2.0])
np.testing.assert_allclose(np.linalg.norm(moved_xyz[1]-moved_xyz[0]), cc_distance_numpy)
np.testing.assert_allclose(bond_angle_deg(moved_xyz[2], moved_xyz[0], moved_xyz[1]), hcc_angle_numpy)
assert abs(angle_difference_deg(dihedral_deg(*moved_xyz[list(torsion_indices)]), torsion_numpy)) < 1e-10
for function, points in [
    (bond_angle_deg, ([0, 0, 0], [0, 0, 0], [1, 0, 0])),
    (dihedral_deg, ([0, 0, 0], [1, 0, 0], [2, 0, 0], [3, 0, 0])),
]:
    try:
        function(*points)
    except ValueError:
        pass  # Expected: the geometry does not define this angle.
    else:
        raise AssertionError("An undefined angle should have been rejected.")
print("Geometry self-checks passed.")

**Try it yourself**

1. Measure every C--H bond and the H2--C0--H3 angle. Why are the lengths and angles similar but not necessarily identical to tabulated experimental values?
2. Set the selected H--C--C--H torsion to 0, 60, and 180 degrees on separate copies. Identify which conformations are eclipsed or staggered. Why is 180 degrees also staggered for ethane?
3. Reflect all coordinates through the $xy$ plane by negating $z$. Predict the effect on distances, bond angles, and the signed dihedral, then check it.
4. Explain why matching only XYZ element symbols cannot verify the ordering of the six hydrogens.

**Answer hints:** (2) 0 is eclipsed; 60 and 180 are staggered for ideal ethane. (3) Distances and bond angles are unchanged; the dihedral sign reverses, up to the $\pm180^\circ$ convention. The force-field chapter will connect these geometries to energies.

### Optional interactive 3D view

Set the flag below to `True` in a trusted Jupyter notebook if you want mouse rotation. py3Dmol normally loads the 3Dmol.js viewer from a CDN, so this optional display may require internet access. The calculations and static figures above do not. Use the explicit format `"mol"` for a Mol block. See the [3Dmol.js API](https://3dmol.org/doc/GLViewer.html#addModel).

In [ ]:
SHOW_INTERACTIVE = False
if SHOW_INTERACTIVE:
    import py3Dmol
    view = py3Dmol.view(width=640, height=400)
    view.addModel(Chem.MolToMolBlock(ethane_mol), "mol")
    view.setStyle({"stick": {}, "sphere": {"scale": 0.25}})
    view.zoomTo()
    view.show()
else:
    print("Interactive viewer disabled; all geometry figures above work offline.")

### References

- [RDKit geometry transformations](https://www.rdkit.org/docs/source/rdkit.Chem.rdMolTransforms.html): angle conventions, units, and coordinate setters.
- [RDKit molecular file readers](https://www.rdkit.org/docs/source/rdkit.Chem.rdmolfiles.html): `MolFromXYZBlock` and coordinate formats.
- [NumPy `arctan2`](https://numpy.org/doc/stable/reference/generated/numpy.arctan2.html): argument order and signed-angle range.

Next: [Part 2 -- force fields and geometry optimization](Chapter02_Part2.ipynb).